# Next Observed Indicator — V4

**Single, coherent, calibrated forecaster** replacing the V3 four-model ensemble.

What changed and why (all validated on a forward-looking backtest):
- **Forward labels.** Trains on *seen in `(t, t+H]`* (the real future), not the trailing window.
- **One monotone model, all horizons.** A single `HistGradientBoostingClassifier` takes the horizon as a feature with a monotonic constraint, so `P(1d) ≤ P(7d) ≤ … ≤ P(45d)` by construction (V4's independent per-horizon models violated this ~41% of the time).
- **Calibrated probabilities.** Because labels are forward, the probabilities mean what they say (ECE ≈ 0.01 vs V4's 0.13 at 30d).
- **Probability-based confidence bands.** The same `Highly likely / Possibly active / Low confidence` labels as V4, but now derived from calibrated probability cutpoints — no hand-tuned `freq ≥ 2` gate (which V4 used and which *hurt* at long horizons).
- **Leakage-safe & no normalization needed.** Features use only data `≤ t`; labels only `(t, t+H]`. Band precision is verified on held-out cutoffs. Tree model needs no feature scaling.


In [ ]:
import os, warnings
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.calibration import CalibratedClassifierCV
warnings.filterwarnings("ignore")

# ------------------------------- CONFIG -------------------------------
OBS_TEMPLATE = "Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d{date}.csv"
DATE_FMT     = "%Y%m%d"

L              = 100          # feature lookback window (days) — same 100-day history as V3, This is how far back featurize() looks to compute behavioral features for any single row
HORIZONS       = [1, 7, 14, 30, 45]
TRAIN_DAYS     = 220          # history pulled to build forward-labeled training data, This is how far back the whole pipeline reaches to assemble the raw event log
CUTOFF_STEP    = 5            # spacing (days) between training cutoffs; must be coprime with 7 (see guard below)
VAL_TAIL_FRAC  = 0.25         # last fraction of training cutoffs used to verify band precision/ data split 75/25

# Catch-the-sightings bands: default High 0.80, per-OpDiv cuts only where
# Precision still holds 90%. Policy lives in noi_v4_bands (same as the runner).
import sys as _sys
_nb_dir = os.path.dirname(os.path.abspath("NextObservedIndicatorV4.0.ipynb"))
if os.path.isdir(_nb_dir) and _nb_dir not in _sys.path:
    _sys.path.insert(0, _nb_dir)
from noi_v4_bands import (  # noqa: E402
    BAND_HIGH_P, BAND_HIGH_P_OPDIV, BAND_LOW_P, BAND_LABELS, band_high, band,
)

INFER_DATE     = None         # None -> use the latest date present in the data
SAVE_OUTPUT    = False        # keep OFF during testing (see final cell)

# cnt(k) features are computed inside an L-day window; keep horizons inside that window.
if max(HORIZONS) > L:
    raise RuntimeError(f"max(HORIZONS)={max(HORIZONS)} exceeds lookback L={L}; cnt(k) would undercount.")

# `dow` is no longer a feature, but a step that is a multiple of 7 still locks
# every cutoff onto one weekday and aliases last_seen / gap structure.
if CUTOFF_STEP % 7 == 0:
    raise RuntimeError(f"CUTOFF_STEP={CUTOFF_STEP} is a multiple of 7; every training cutoff "
                       f"would lock onto one weekday and alias last_seen / gap structure.")

EPOCH = np.datetime64("2020-01-01")
def _to_int(d): return (np.asarray(d, dtype="datetime64[D]") - EPOCH).astype(int)
def _to_ts(i):  return pd.Timestamp(EPOCH + np.timedelta64(int(i), "D")) #converts dates to integer day numbers.


In [8]:
def load_panel(days):
    """Load the daily 'seen' panel (indicator, opdiv, date) over the last `days` days."""
    today = datetime.today().date()
    start = today - timedelta(days=days)
    frames, d = [], start
    while d <= today:
        fp = OBS_TEMPLATE.format(date=d.strftime(DATE_FMT))
        if os.path.exists(fp):
            try:
                frames.append(pd.read_csv(fp, usecols=["indicator", "obs_date", "OpDiv"]))
            except Exception as e:
                print("skip", fp, e)
        d += timedelta(days=1)
    if not frames:
        raise RuntimeError(
            f"No observation files loaded for {start} -> {today} "
            f"(template={OBS_TEMPLATE}). Check share path / date coverage."
        )
    p = pd.concat(frames, ignore_index=True)
    p["indicator"] = p["indicator"].astype(str).str.strip()
    p["opdiv"]     = p["OpDiv"].astype(str).str.strip()
    p["date"]      = pd.to_datetime(p["obs_date"], errors="coerce").dt.normalize()
    p = p[["indicator", "opdiv", "date"]].dropna()
    p = p[p["indicator"].ne("nan") & p["indicator"].ne("")]
    p = p.drop_duplicates(["indicator", "opdiv", "date"])
    if p.empty:
        raise RuntimeError("Observation files were found, but the cleaned panel is empty.")
    p["d"] = _to_int(p["date"].values.astype("datetime64[D]"))
    return p

panel = load_panel(TRAIN_DAYS)
# per (opdiv, indicator) -> sorted array of int-days seen (fast window queries)
lookup = {}
for (opd, ind), g in panel.groupby(["opdiv", "indicator"], sort=False):
    lookup[(opd, ind)] = np.sort(g["d"].to_numpy())
DAY_MIN, DAY_MAX = int(panel["d"].min()), int(panel["d"].max())
print(f"panel: {len(panel):,} rows | {len(lookup):,} (opdiv,indicator) | "
      f"{_to_ts(DAY_MIN).date()} -> {_to_ts(DAY_MAX).date()}")


panel: 845,024 rows | 25,040 (opdiv,indicator) | 2025-12-16 -> 2026-07-24


In [ ]:
FEATS = ["last_seen", "freq_7", "freq_14", "freq_30", "freq_100",
         "avg_gap", "burstiness", "overdue"]
# Dropped after held-out permutation (31 Aug 2026): freq_1, freq_45, active_frac,
# dow, mom, tenure, weekday_hit, skip_gap_frac. last_seen is ~73% of AUC drop;
# the rest of this list is the cadence cluster that still paid rent.

def featurize(dates, t):
    """Behavioral features from observations <= t (recency, frequency, burst, cadence).
    Leakage-safe: every value uses only observations on or before t."""
    hi = np.searchsorted(dates, t, side="right")
    lo = np.searchsorted(dates, t - L + 1, side="left")
    win = dates[lo:hi]
    if win.size == 0:
        return [L, 0, 0, 0, 0, float(L), 0.0, 1.0]
    def cnt(k):
        return int(win.size - np.searchsorted(win, t - k + 1, side="left"))
    gaps = np.diff(win)
    if gaps.size >= 1:
        ag = float(gaps.mean()); sd = float(gaps.std())
        bu = (sd - ag) / (sd + ag) if (sd + ag) > 0 else 0.0
    else:
        ag, bu = float(L), 0.0
    last_seen = int(t - win[-1])
    overdue = last_seen / ag if ag > 0 else 0.0
    return [last_seen, cnt(7), cnt(14), cnt(30), win.size, ag, bu, overdue]

def seen_next(dates, t, H):
    """Forward label: 1 if observed on any day in (t, t+H]."""
    return 1 if np.searchsorted(dates, t + H, side="right") > np.searchsorted(dates, t + 1, side="left") else 0

def build_rows(cutoffs, need_label=True):
    """One row per (opdiv, indicator, cutoff) for indicators active within the lookback window."""
    recs = []
    for t in cutoffs:
        for (opd, ind), dates in lookup.items():
            hi = np.searchsorted(dates, t, side="right")
            lo = np.searchsorted(dates, t - L + 1, side="left")
            if hi - lo == 0:          # candidate = seen at least once in the lookback
                continue
            rec = dict(zip(FEATS, featurize(dates, t)))
            rec["opdiv"], rec["indicator"], rec["t"] = opd, ind, t
            if need_label:
                for H in HORIZONS:
                    rec[f"y_{H}"] = seen_next(dates, t, H)
            recs.append(rec)
    return pd.DataFrame(recs)


In [10]:
# Training cutoffs: every CUTOFF_STEP days, with a full lookback AND a matured max-horizon label.
maxH = max(HORIZONS)
train_cutoffs = list(range(DAY_MIN + L, DAY_MAX - maxH + 1, CUTOFF_STEP))
if not train_cutoffs:
    raise RuntimeError(
        f"Not enough history for training cutoffs "
        f"(DAY_MIN={DAY_MIN}, DAY_MAX={DAY_MAX}, L={L}, maxH={maxH}). "
        f"Increase TRAIN_DAYS."
    )

# Hold out the most-recent VAL_TAIL_FRAC of cutoffs to verify the bands OUT-OF-SAMPLE (no leakage).
# model_val (fit on earlier cutoffs) is scored in the band/dormancy diagnostic cells below.
n_val = max(1, int(len(train_cutoffs) * VAL_TAIL_FRAC))
val_cutoffs = set(train_cutoffs[-n_val:]) if len(train_cutoffs) > n_val else set()
train_df = build_rows(train_cutoffs, need_label=True)
val_df = train_df[train_df["t"].isin(val_cutoffs)].copy()     # held out from the verification model
fit_df = train_df[~train_df["t"].isin(val_cutoffs)].copy()    # earlier cutoffs only
if fit_df.empty or val_df.empty:
    raise RuntimeError(
        f"Validation split produced an empty frame "
        f"(fit={len(fit_df):,}, held-out={len(val_df):,}, cutoffs={len(train_cutoffs)})."
    )
print(f"training rows: {len(train_df):,} (fit {len(fit_df):,} / held-out {len(val_df):,}) "
      f"from {len(train_cutoffs)} cutoffs "
      f"({_to_ts(train_cutoffs[0]).date()} -> {_to_ts(train_cutoffs[-1]).date()})")

def stack(df, with_label=True, with_weights=False):
    """Stack once per horizon, appending horizon as a feature -> ONE model for all horizons."""
    base = df[FEATS].to_numpy(float)
    Xs, ys, ws = [], [], []
    for H in HORIZONS:
        Xh = np.hstack([base, np.full((len(df), 1), H, float)])
        if not with_label:
            Xs.append(Xh)
            continue
        y = df[f"y_{H}"].to_numpy()
        Xs.append(Xh)
        ys.append(y)
        if with_weights:
            ws.append(_balanced_group_weights(df["opdiv"].to_numpy(), y))
    X = np.vstack(Xs)
    yout = np.concatenate(ys) if with_label else None
    if with_label and with_weights:
        return X, yout, np.concatenate(ws)
    return X, yout

def _balanced_group_weights(opdivs, y):
    """Balance positives and negatives inside each OpDiv for this horizon slice."""
    opdivs = np.asarray(opdivs)
    y = np.asarray(y).astype(int)
    w = np.ones(len(y), dtype=float)
    for g in np.unique(opdivs):
        m = opdivs == g
        n = int(m.sum())
        n_pos = int(y[m].sum())
        n_neg = n - n_pos
        if n_pos == 0 or n_neg == 0:
            continue
        w[m & (y == 1)] = n / (2.0 * n_pos)
        w[m & (y == 0)] = n / (2.0 * n_neg)
    return w

# NORMALIZATION: none needed. HistGBT is tree-based -> invariant to monotonic feature
# transforms, so scaling/standardizing changes nothing; it also handles NaN natively.
# Features are used as-is. (V4 scaled only because it fed LogisticRegression; a linear
# model WOULD need StandardScaler.)
mono = [0] * len(FEATS) + [1]     # probability must be non-decreasing in the horizon feature

# early_stopping is OFF deliberately. sklearn's default ('auto') turns it on above
# 10k rows and holds out a RANDOM 10%, but stack() emits each (opdiv, indicator, t)
# once per horizon and cutoffs are only CUTOFF_STEP days apart, so a row's own
# near-duplicates end up on the training side of that split. The internal score is
# then optimistic and the stopping point is meaningless. Capacity is bounded by
# max_iter / max_depth / l2 instead, which also makes the tree count reproducible.
def new_model():
    return HistGradientBoostingClassifier(
        max_depth=4, learning_rate=0.08, max_iter=400, l2_regularization=1.0,
        monotonic_cst=mono, early_stopping=False, random_state=0)

# OpDiv-balanced training on earlier cutoffs, isotonic calibration on the held-out
# tail so p still means p. Same recipe as the scheduled runner.
Xf, yf, wf = stack(fit_df, with_weights=True)
base = new_model().fit(Xf, yf, sample_weight=wf)
Xv, yv = stack(val_df)
model = CalibratedClassifierCV(base, method="isotonic", cv="prefit").fit(Xv, yv)
model_val = model
print(
    f"trained OpDiv-balanced model, isotonic-calibrated on held-out tail "
    f"(val pos rate {yv.mean() * 100:.1f}%, mean predicted p {model.predict_proba(Xv)[:, 1].mean() * 100:.1f}%)"
)


training rows: 141,151 (fit 112,951 / held-out 28,200) from 11 cutoffs (2026-03-26 -> 2026-06-04)
trained verification model (fit cutoffs) + production model (all cutoffs)


In [11]:
# Band verification on the HELD-OUT cutoffs. model_val was NOT trained on these rows,
# so the precision figures below are out-of-sample (leakage-free).
print("Band verification on HELD-OUT cutoffs (out-of-sample):")
print(f"{'H':>3} | {'High n':>7} {'High prec':>9} {'High recall':>11} | "
      f"{'Poss n':>8} {'Poss hit':>9} | {'Low n':>7} {'Low neg-prec':>12}")
for H in HORIZONS:
    xb = np.hstack([val_df[FEATS].to_numpy(float), np.full((len(val_df), 1), H, float)])
    pv = model_val.predict_proba(xb)[:, 1]
    yv = val_df[f"y_{H}"].to_numpy()
    hi = pv >= band_high(H); lo = pv <= BAND_LOW_P; wa = (~hi) & (~lo)
    hp = yv[hi].mean() if hi.sum() else np.nan
    hr = yv[hi].sum() / yv.sum() if yv.sum() else np.nan
    wh = yv[wa].mean() if wa.sum() else np.nan
    ln = (1 - yv[lo]).mean() if lo.sum() else np.nan
    print(f"{H:>3} | {int(hi.sum()):>7,} {hp:>9.3f} {hr:>11.3f} | "
          f"{int(wa.sum()):>8,} {wh:>9.3f} | {int(lo.sum()):>7,} {ln:>12.3f}")


Band verification on HELD-OUT cutoffs (out-of-sample):
  H |  High n High prec High recall |   Poss n  Poss hit |   Low n Low neg-prec
  1 |   4,896     0.932       0.695 |    2,616     0.461 |  20,688        0.961
  7 |   6,710     0.974       0.664 |    4,069     0.561 |  17,421        0.941
 14 |   7,364     0.972       0.643 |    4,899     0.603 |  15,937        0.936
 30 |   8,190     0.973       0.657 |    6,463     0.512 |  13,547        0.938
 45 |   8,548     0.974       0.657 |    6,589     0.511 |  13,063        0.926


In [ ]:
eval_cutoffs = sorted(val_cutoffs) if val_cutoffs else list(train_cutoffs)

# ---- (A) recall by dormancy bucket, among scored candidates (seen within L) ----
diag = build_rows(eval_cutoffs, need_label=True)
Xd, _ = stack(diag, with_label=False)
Pd = model_val.predict_proba(Xd)[:, 1].reshape(len(HORIZONS), len(diag)).T
Pd = np.maximum.accumulate(Pd, axis=1)                       # same monotone step as inference
gap = diag["last_seen"].to_numpy()                           # days since last seen (0..L-1 for candidates)

BUCKETS = [(0, 0, "seen today"), (1, 7, "1-7d"), (8, 30, "8-30d"),
           (31, 60, "31-60d"), (61, L - 1, f"61-{L-1}d")]
print("(A) Scored candidates -- 'Highly likely' RECALL among true recurrences, by days-since-last-seen")
print(f"{'gap bucket':>12} | " + "  ".join(f"H{H:>2}" for H in HORIZONS) + "   | recur@30d")
for lo, hi, name in BUCKETS:
    m = (gap >= lo) & (gap <= hi)
    cells, n30 = [], 0
    for j, H in enumerate(HORIZONS):
        y = diag[f"y_{H}"].to_numpy()[m]
        rec = (Pd[m, j] >= band_high(H))[y == 1].mean() if (y == 1).any() else np.nan
        cells.append(f"{rec:4.2f}")
        if H == 30:
            n30 = int((y == 1).sum())
    print(f"{name:>12} | " + "  ".join(cells) + f"   | {n30:>8,}")

# ---- (B) structural blind spot: recurrences the pipeline never scores ----
tot = {H: 0 for H in HORIZONS}; dorm = {H: 0 for H in HORIZONS}; cold = {H: 0 for H in HORIZONS}
for t in eval_cutoffs:
    for dates in lookup.values():
        i_t  = int(np.searchsorted(dates, t, side="right"))         # # obs <= t
        i_tL = int(np.searchsorted(dates, t - L + 1, side="left"))  # first idx >= t-L+1
        seen_before = i_t > 0
        seen_L      = (i_t - i_tL) > 0                              # any obs in the prior L days
        for H in HORIZONS:
            if seen_next(dates, t, H):
                tot[H] += 1
                if not seen_L:
                    (dorm if seen_before else cold)[H] += 1

print("\n(B) Share of TRUE recurrences from indicators the candidate filter NEVER scores")
print(f"{'H':>3} | {'total recur':>11} | {'dormant':>10} {'(>%dd silent)':>13}".replace("%d", str(L))
      + f" | {'cold-start':>10} {'(never seen)':>13}")
for H in HORIZONS:
    d = dorm[H] / tot[H] if tot[H] else float("nan")
    c = cold[H] / tot[H] if tot[H] else float("nan")
    print(f"{H:>3} | {tot[H]:>11,} | {dorm[H]:>10,} {d:>12.1%} | {cold[H]:>10,} {c:>12.1%}")

print("\nRead: (A) how fast our reliable signal fades as an indicator goes quiet -- the recall we"
      "\n       could still recover from the observation stream (cadence/silence features).")
print("      (B) the recurrence we CANNOT touch without external data -- dormant reactivations and"
      "\n       cold starts have no signal in the observation timeline; that fraction is the hard ceiling.")


In [ ]:
# ---- Inference: forecast for the as-of date (no label; this is the real forecast) ----
infer_t = DAY_MAX if INFER_DATE is None else int(_to_int(np.datetime64(pd.Timestamp(INFER_DATE).date())))
infer_df = build_rows([infer_t], need_label=False)
print(f"scoring {len(infer_df):,} indicators as-of {_to_ts(infer_t).date()}")
if infer_df.empty:
    raise RuntimeError(f"No candidate indicators to score as-of {_to_ts(infer_t).date()}.")

Xinf, _ = stack(infer_df, with_label=False)
P = model.predict_proba(Xinf)[:, 1].reshape(len(HORIZONS), len(infer_df)).T   # rows=indicators, cols=horizons
P = np.maximum.accumulate(P, axis=1)                                          # enforce monotonicity across horizons

def band(p, H, opdiv=None):
    if p >= band_high(H, opdiv): return BAND_LABELS["H"]
    if p <= BAND_LOW_P:  return BAND_LABELS["L"]
    return BAND_LABELS["W"]

out = infer_df[["opdiv", "indicator", "last_seen", "freq_7", "freq_30"]].copy()
for j, H in enumerate(HORIZONS):
    out[f"prob_{H}"] = P[:, j]
    out[f"band_{H}"] = [band(p, H, opd) for p, opd in zip(P[:, j], out["opdiv"])]

# ---- Format to the existing production schema (drop-in with downstream tooling) ----
PROBNAME = {1: "Probability: 1-Day", 7: "Probability: 7-Day", 14: "Probability: 14-Day",
            30: "Probability: 30-Day", 45: "Probability: 45-Day"}
def to_production(g):
    d = pd.DataFrame({
        "Indicator":       g["indicator"].values,
        "Observed Today":  (g["last_seen"].values == 0).astype(int),
        "Frequency (1d)":  (g["last_seen"].values == 0).astype(int),
        "Frequency (7d)":  g["freq_7"].values,
        "Frequency (30d)": g["freq_30"].values,
    })
    for H in HORIZONS:
        d[PROBNAME[H]] = (g[f"prob_{H}"].values * 100).round(2).astype(str) + "%"
        d[f"Confidence: {H}-Day"] = [f"{H}-Day: {b}" for b in g[f"band_{H}"].values]
    cols = ["Indicator", "Observed Today", "Frequency (1d)", "Frequency (7d)", "Frequency (30d)"]
    for H in [1, 7, 14, 30]:
        cols += [f"Probability: {H}-Day", f"Confidence: {H}-Day"]
    cols += ["Probability: 45-Day", "Confidence: 45-Day"]
    return d[cols]

opdiv_outputs = {opd: to_production(g).reset_index(drop=True) for opd, g in out.groupby("opdiv")}
print("OpDivs:", list(opdiv_outputs.keys()))
display(opdiv_outputs[list(opdiv_outputs)[0]].head(10))


scoring 16,729 indicators as-of 2026-07-24
OpDivs: ['CDC', 'CMS', 'DHA', 'FDA', 'HHS', 'HRSA', 'IHS', 'NIH', 'OS', 'VA']


,Indicator,Observed Today,Frequency (1d),Frequency (7d),Frequency (30d),Probability: 1-Day,Confidence: 1-Day,Probability: 7-Day,Confidence: 7-Day,Probability: 14-Day,Confidence: 14-Day,Probability: 30-Day,Confidence: 30-Day,ensemble_45d,Confidence: 45-Day
0,64.62.156.109,0,0,0,0,0.34%,1-Day: Low confidence,2.33%,7-Day: Low confidence,4.16%,14-Day: Low confidence,7.35%,30-Day: Low confidence,9.71%,45-Day: Low confidence
1,64.62.156.33,0,0,0,0,0.36%,1-Day: Low confidence,2.28%,7-Day: Low confidence,4.08%,14-Day: Low confidence,7.08%,30-Day: Low confidence,9.37%,45-Day: Low confidence
2,64.62.156.46,0,0,0,0,0.3%,1-Day: Low confidence,1.92%,7-Day: Low confidence,3.91%,14-Day: Low confidence,6.8%,30-Day: Low confidence,9.01%,45-Day: Low confidence
3,64.62.156.71,0,0,0,1,1.47%,1-Day: Low confidence,7.62%,7-Day: Low confidence,13.95%,14-Day: Low confidence,22.26%,30-Day: Possibly active,26.35%,45-Day: Possibly active
4,64.62.156.75,0,0,0,0,0.34%,1-Day: Low confidence,2.37%,7-Day: Low confidence,4.36%,14-Day: Low confidence,7.83%,30-Day: Low confidence,10.34%,45-Day: Low confidence
5,65.49.1.45,0,0,0,0,0.34%,1-Day: Low confidence,2.37%,7-Day: Low confidence,4.36%,14-Day: Low confidence,7.83%,30-Day: Low confidence,10.34%,45-Day: Low confidence
6,74.208.236.241,0,0,0,5,7.16%,1-Day: Low confidence,39.09%,7-Day: Possibly active,59.78%,14-Day: Possibly active,71.8%,30-Day: Possibly active,76.15%,45-Day: Possibly active
7,geo.netsupportsoftware.com/location/loca.asp,0,0,2,4,9.23%,1-Day: Low confidence,41.43%,7-Day: Possibly active,58.44%,14-Day: Possibly active,73.37%,30-Day: Possibly active,78.12%,45-Day: Possibly active
8,onestart.ai/,0,0,3,10,24.37%,1-Day: Possibly active,77.4%,7-Day: Possibly active,87.79%,14-Day: Highly likely,91.86%,30-Day: Highly likely,94.01%,45-Day: Highly likely
9,www.shorturl.at/,0,0,3,13,17.54%,1-Day: Low confidence,72.52%,7-Day: Possibly active,83.22%,14-Day: Highly likely,88.63%,30-Day: Highly likely,91.54%,45-Day: Highly likely


: 

In [ ]:
# ------------------------------------------------------------------
# OPTIONAL SAVE — disabled by default. Do NOT point this at production
# prediction folders during testing. Uses a separate dev path.
# ------------------------------------------------------------------
if SAVE_OUTPUT:
    # Test path (NOT production OpDiv_Predictions). UNC-safe for Task Scheduler.
    SAVE_DIR = r"\\10.1.4.22\data\HTOC\JA\NextObserveV4Test"
    stamp = _to_ts(infer_t).strftime("%Y%m%d")
    os.makedirs(SAVE_DIR, exist_ok=True)
    for opd, df_out in opdiv_outputs.items():
        sub = os.path.join(SAVE_DIR, opd)
        os.makedirs(sub, exist_ok=True)
        df_out.to_csv(os.path.join(sub, f"{opd}_output_{stamp}.csv"), index=False)
    print("saved to", SAVE_DIR)
else:
    print("SAVE_OUTPUT is False - nothing written.")


## Notes / productionizing

- **Leakage & normalization.** Features use only observations `≤ t`; labels only `(t, t+H]`; no indicator/OpDiv identity is a feature. The verification model is trained on earlier cutoffs and scored on held-out later cutoffs, so the band precision below is out-of-sample. No feature scaling is applied — a gradient-boosted tree is invariant to it.
- **Bands are trustworthy by verification.** On held-out cutoffs: *Highly likely* runs ~93–97% precision, *Low confidence* ~93–96% correct-negative, and *Possibly active* is an honest ~50% — the genuinely ambiguous middle (mostly indicators last seen 2–30 days ago). Backtesting showed this middle is largely irreducible from the observation stream; shrinking it needs external signal (threat-intel / indicator attributes), not more modeling.
- **What we can confidently forecast.** Continuation of *known* activity, as early as 1 day out. Brand-new (cold-start) and long-dormant indicators are near-unpredictable at any horizon — a data limit, not a model limit.
- **Efficiency.** This retrains one model each run from a 220-day pull (~1–2 min I/O). For production, persist `model` + the config after a periodic (e.g., monthly) refit and pull only recent days for daily inference.
- **CDC caveat.** A single global model slightly regresses on CDC (its hardest, noisiest OpDiv). If that matters, train **per-OpDiv** models (or add OpDiv as a categorical feature) — the loop structure already groups by OpDiv.
- **Tuning the bands.** Mission is catch-the-sightings: maximize recall vs all positives, keep precision ≥ 90%. Default High is 0.80. `noi_v4_bands.BAND_HIGH_P_OPDIV` lowers High only for OpDivs that still hold the floor (1-day: CMS 0.65, HRSA 0.75; 7-day: CMS/OS/FDA/HHS/NIH 0.65, HRSA/VA/CDC 0.70). OS/CDC/DHA/HHS/NIH/IHS at 1-day are already at or under the floor — do not cut them. A global 0.50 cut flooded false positives; do not use it. Do not promote Possibly Active into 1-day High (`p≥0.70` or 7-day High as a 1-day proxy). Low stays at 0.20.
- **Three jobs, three bands.** 1-day High is the tomorrow page (metronomes, ~90% precise). 7-day High is the weekly board (skip-day regulars). Possibly Active is abstain, not a worklist. Judge skip-day coverage on 7-day High, not by stuffing those rows into 1-day High.
